# Adult (PATE-GAN) 

Author: Ilse Harmers \
Last modified: February 17, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from snsynth import Synthesizer
import snsynth.transform as tf

In [ ]:
# Importing train set.
adult_train = pd.read_csv("./train-test-datasets/adult/adult_train.csv")
print(adult_train.columns.to_list())
adult_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = adult_train["age"].min(), upper = adult_train["age"].max(), 
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # workclass
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education-num
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital-status
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # occupation
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # relationship
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # race
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # sex
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(adult_train["capital-gain"].max() + 1),
                             negative = False) # capital-gain; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(4400.0 + 1),   # 1 id with 4356.0; rounded up w.r.t. privacy
                             negative = False) # capital-loss; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = adult_train["hours-per-week"].min(), 
                         upper = adult_train["hours-per-week"].max(), 
                         negative = False), # hours-per-week; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # native-country
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # income
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 4 * 10^4 rows, then delta = 10^(-5). 
delta = 10**np.floor(np.log10(1/adult_train.shape[0]))
print(delta)

In [ ]:
# Synthesizing the dataset with PATE-GAN. 
synth = Synthesizer.create('pategan', epsilon = 1.0, delta = delta, plot_losses = True)
synth.fit(adult_train, transformer = tt, preprocessor_eps = 0.0)

In [ ]:
# Generating a synthetic dataset with the same amount of rows as the original training dataset.
sample = synth.sample(adult_train.shape[0])

In [ ]:
sample.describe()

In [ ]:
# Saving the synthetic dataset as a csv file.
sample.to_csv("./synthetic-datasets_PATE-GAN/adult/epsi_1/run5/sample3.csv", index = False)